In [ ]:
import pydicom
import pandas as pd
import glob

def get_dicom_metadata_df(file_paths):
    """
    Read metadata from a list of DICOM file paths
    and return a sorted DataFrame.
    """
    rows = []

    for path in file_paths:
        ds = pydicom.dcmread(path, stop_before_pixels=True)

        row = {
            "file_path": path,
            # IDs (to link slices back to patient/study/series)
            "PatientID": getattr(ds, "PatientID", None),
            "StudyInstanceUID": getattr(ds, "StudyInstanceUID", None),
            "SeriesInstanceUID": getattr(ds, "SeriesInstanceUID", None),
            "SOPInstanceUID": getattr(ds, "SOPInstanceUID", None),

            # Order info (most important part)
            "SeriesNumber": getattr(ds, "SeriesNumber", None),
            "InstanceNumber": getattr(ds, "InstanceNumber", None),
            # "SliceLocation": getattr(ds, "SliceLocation", None),
            

            # Useful features for the model
            "SeriesDescription": getattr(ds, "SeriesDescription", None),
            "BodyPartExamined": getattr(ds, "BodyPartExamined", None),
            "Laterality": getattr(ds, "Laterality", None),
            "PatientSex": getattr(ds, "PatientSex", None),
            "ScanningSequence": getattr(ds, "ScanningSequence", None),
            # "SequenceVariant": getattr(ds, "SequenceVariant", None),
            # "ScanOptions": getattr(ds, "ScanOptions", None),
            "MRAcquisitionType": getattr(ds, "MRAcquisitionType", None),
            # "SliceThickness": getattr(ds, "SliceThickness", None),
            # "SpacingBetweenSlices": getattr(ds, "SpacingBetweenSlices", None),
            # "PixelSpacing": getattr(ds, "PixelSpacing", None),
            "Rows": getattr(ds, "Rows", None),
            "Columns": getattr(ds, "Columns", None),
            # "RepetitionTime": getattr(ds, "RepetitionTime", None),
            # "EchoTime": getattr(ds, "EchoTime", None),
            "MagneticFieldStrength": getattr(ds, "MagneticFieldStrength", None),
            "FlipAngle": getattr(ds, "FlipAngle", None),
            "Manufacturer": getattr(ds, "Manufacturer", None),
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    # convert to numeric so sorting works correctly
    df["InstanceNumber"] = pd.to_numeric(df["InstanceNumber"], errors="coerce")

    # sort: group by series, then order slices inside each series
    df = df.sort_values(
        by=["SeriesInstanceUID", "InstanceNumber", ]
    ).reset_index(drop=True)

    return df



def group_dicom_by_series(dicom_df: pd.DataFrame) -> pd.DataFrame:
    """One row per series. Constant columns keep one value, others become lists."""
    value_cols = [c for c in dicom_df.columns if c not in GROUP_KEYS]
    unique_counts = dicom_df.groupby(GROUP_KEYS)[value_cols].nunique(dropna=False)
    constant_cols = set(unique_counts.columns[unique_counts.max() <= 1])

    agg_dict = {c: "first" if c in constant_cols else list for c in value_cols}
    return dicom_df.groupby(GROUP_KEYS, as_index=False).agg(agg_dict)


def build_merged_df(train_series_path: str, dicom_df: pd.DataFrame) -> pd.DataFrame:
    """Merge series labels with grouped DICOM metadata and clean it."""
    train_series = pd.read_csv(train_series_path)
    dicom_grouped = group_dicom_by_series(dicom_df)

    df = train_series.merge(dicom_grouped, on=GROUP_KEYS, how="left")

    # Standardize names
    df["Laterality"] = df["Laterality"].replace(LATERALITY_MAP)
    df["Manufacturer"] = df["Manufacturer"].replace(MANUFACTURER_MAP)

    # Drop columns you don't need (ignore if missing)
    df = df.drop(columns=DROP_COLS, errors="ignore")

    # Keep the highest instance number per series (only if it is a list)
    df["InstanceNumber"] = df["InstanceNumber"].apply(
        lambda x: max(x) if isinstance(x, list) else x
    )
    return df



In [ ]:
GROUP_KEYS = ["StudyInstanceUID", "SeriesInstanceUID"]

LATERALITY_MAP = {"R": "RIGHT", "L": "LEFT"}

MANUFACTURER_MAP = {
    "SIEMENS": "Siemens",
    "Siemens Healthineers": "Siemens",
    "Philips Medical Systems": "Philips",
    "Philips Healthcare": "Philips",
    "GE MEDICAL SYSTEMS": "GE",
    "GEHC": "GE",
    "CANON_MEC": "Canon",
    "TOSHIBA": "Toshiba",
    "FUJIFILM Healthcare Corporation": "Fujifilm",
    "Hitachi Medical Corporation": "Hitachi",
}

DROP_COLS = ["SOPInstanceUID", "SeriesNumber", "PatientID"]



In [ ]:

file_paths = glob.glob("/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series/**/*.dcm", recursive=True)

print(f"Found {len(file_paths)} dicom files")

df = get_dicom_metadata_df(file_paths)
df.head()
df.to_csv('df_metadata.csv', index = False)

df_merged = build_merged_df(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv",
    df, 
)

for col in ["Laterality", "Manufacturer"]:
    print(col, ":", df_merged[col].unique())

df_merged.to_csv("df_merged.csv", index=False)
